In [23]:
import os
import random
import optuna
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler


from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor





def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
print(f"Исходные данные загружены. Train: {train.shape}, Test: {test.shape}")


Исходные данные загружены. Train: (751, 214), Test: (250, 211)


In [3]:
target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in ["index"] + target_cols]


In [4]:
medians = train.groupby(feature_cols)[target_cols].transform('median')
train[target_cols] = medians
train_cleaned = train.drop_duplicates(subset=feature_cols, keep='first').reset_index(drop=True)
print(f"После объединения дубликатов молекул по медиане: {train_cleaned.shape}")


После объединения дубликатов молекул по медиане: (630, 214)


In [5]:
train_cleaned = train_cleaned.dropna(subset=target_cols).reset_index(drop=True)
print(f"После удаления строк с пустыми таргетами (NaN): {train_cleaned.shape}")


После удаления строк с пустыми таргетами (NaN): (628, 214)


In [6]:
selector = VarianceThreshold(threshold=0.0)
selector.fit(train_cleaned[feature_cols])
constant_features = [col for col, keep in zip(feature_cols, selector.get_support()) if not keep]
feature_cols = [col for col in feature_cols if col not in constant_features]
print(f"Удалено константных признаков: {len(constant_features)}. Осталось признаков: {len(feature_cols)}")


Удалено константных признаков: 18. Осталось признаков: 192


In [7]:
corr_matrix = train_cleaned[feature_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]
feature_cols = [col for col in feature_cols if col not in high_corr_features]
print(f"Удалено сильно коррелирующих признаков (>0.95): {len(high_corr_features)}. Итого признаков: {len(feature_cols)}")


Удалено сильно коррелирующих признаков (>0.95): 34. Итого признаков: 158


In [8]:
q25 = train_cleaned['SI'].quantile(0.25)
q75 = train_cleaned['SI'].quantile(0.75)
iqr = q75 - q25
upper_boundary = q75 + 3.0 * iqr
train_final = train_cleaned[train_cleaned['SI'] <= upper_boundary].reset_index(drop=True)
print(f"Граница выбросов по IQR для SI: {upper_boundary:.2f}. Удалено выбросов: {train_cleaned.shape[0] - train_final.shape[0]}")


Граница выбросов по IQR для SI: 48.59. Удалено выбросов: 49


In [9]:
train_final['fold'] = -1
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_final)):
    train_final.loc[val_idx, 'fold'] = fold_idx

print(f"\n Очищенный датасет")
print(f"Размерность train_final: {train_final.shape}")
print(f"Количество признаков в feature_cols: {len(feature_cols)}")
print(f"Распределение строк по 5 фолдам:\n{train_final['fold'].value_counts().to_string()}")


 Очищенный датасет
Размерность train_final: (579, 215)
Количество признаков в feature_cols: 158
Распределение строк по 5 фолдам:
fold
1    116
0    116
2    116
3    116
4    115


C:\Users\Professional\AppData\Local\Temp\ipykernel_15684\3667698995.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_final['fold'] = -1


In [10]:
train_final = train_final.copy()


In [11]:
models_ic50 = []
models_cc50 = []
models_si = []

train_final['oof_IC50'] = 0.0
train_final['oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_cc50.append(model)

feature_cols_si = feature_cols + ['oof_IC50', 'oof_CC50']
for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols_si], train_final.loc[train_idx, 'SI'],
              eval_set=(train_final.loc[val_idx, feature_cols_si], train_final.loc[val_idx, 'SI']),
              early_stopping_rounds=100, verbose=False)
    models_si.append(model)

test_preds_ic50 = np.zeros(len(test))
test_preds_cc50 = np.zeros(len(test))
test_preds_si = np.zeros(len(test))

for model in models_ic50:
    test_preds_ic50 += model.predict(test[feature_cols]) / 5

for model in models_cc50:
    test_preds_cc50 += model.predict(test[feature_cols]) / 5

test_meta = test.copy()
test_meta['oof_IC50'] = test_preds_ic50
test_meta['oof_CC50'] = test_preds_cc50

for model in models_si:
    test_preds_si += model.predict(test_meta[feature_cols_si]) / 5


submission = pd.DataFrame({
    'index': test['index'],
    'IC50': test_preds_ic50,
    'CC50': test_preds_cc50,
    'SI': test_preds_si
})

submission.to_csv('submission.csv', index=False)
print("Файл submission.csv создан. Формат:")
print(submission.head())
print(f"\nРазмерность файла: {submission.shape}")

Файл submission.csv создан. Формат:
   index        IC50        CC50        SI
0      0  173.109074  360.244931  8.202102
1      1  235.091709  379.337866  5.609795
2      2  158.658089  305.884217  8.568503
3      3  281.977935  396.177885  6.447149
4      4  207.574996  357.836030  4.830452

Размерность файла: (250, 4)


In [12]:
xgb_models_ic50 = []
xgb_models_cc50 = []
xgb_models_si = []

train_final['xgb_oof_IC50'] = 0.0
train_final['xgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_cc50.append(model)

feature_cols_xgb_si = feature_cols + ['xgb_oof_IC50', 'xgb_oof_CC50']
xgb_oof_predictions_si = np.zeros(len(train_final))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols_xgb_si], train_final.loc[train_idx, 'SI'],
        eval_set=[(train_final.loc[val_idx, feature_cols_xgb_si], train_final.loc[val_idx, 'SI'])],
        verbose=False
    )
    xgb_oof_predictions_si[val_idx] = model.predict(train_final.loc[val_idx, feature_cols_xgb_si])
    xgb_models_si.append(model)

xgb_test_ic50 = np.zeros(len(test))
xgb_test_cc50 = np.zeros(len(test))
xgb_test_si = np.zeros(len(test))

for model in xgb_models_ic50:
    xgb_test_ic50 += model.predict(test[feature_cols]) / 5

for model in xgb_models_cc50:
    xgb_test_cc50 += model.predict(test[feature_cols]) / 5

test_meta_xgb = test.copy()
test_meta_xgb['xgb_oof_IC50'] = xgb_test_ic50
test_meta_xgb['xgb_oof_CC50'] = xgb_test_cc50

for model in xgb_models_si:
    xgb_test_si += model.predict(test_meta_xgb[feature_cols_xgb_si]) / 5

print("Обучение XGBoost завершено")
print(f"Локальный OOF RMSE для IC50 (XGB): {root_mean_squared_error(train_final['IC50, mM'], train_final['xgb_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (XGB): {root_mean_squared_error(train_final['CC50, mM'], train_final['xgb_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (XGB): {root_mean_squared_error(train_final['SI'], xgb_oof_predictions_si):.4f}")


Обучение XGBoost завершено
Локальный OOF RMSE для IC50 (XGB): 330.7210
Локальный OOF RMSE для CC50 (XGB): 451.3674
Локальный OOF RMSE для SI (XGB): 9.5953


In [13]:
blended_ic50 = (test_preds_ic50 + xgb_test_ic50) / 2
blended_cc50 = (test_preds_cc50 + xgb_test_cc50) / 2
blended_si = (test_preds_si + xgb_test_si) / 2
submission_blended = pd.DataFrame({
    'index': test['index'],
    'IC50': blended_ic50,
    'CC50': blended_cc50,
    'SI': blended_si
})

submission_blended.to_csv('submission_blended.csv', index=False)

print("Файл submission_blended.csv создан")
print(submission_blended.head())

Файл submission_blended.csv создан
   index        IC50        CC50        SI
0      0  183.376287  407.301821  8.574248
1      1  237.705301  390.244605  5.849062
2      2  153.684235  346.939434  8.095828
3      3  303.942306  438.748698  6.133202
4      4  198.019498  353.846655  5.372371


In [14]:
lgb_models_ic50 = []
lgb_models_cc50 = []
lgb_models_si = []

train_final['lgb_oof_IC50'] = 0.0
train_final['lgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    train_final.loc[val_idx, 'lgb_oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    lgb_models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    train_final.loc[val_idx, 'lgb_oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    lgb_models_cc50.append(model)

feature_cols_lgb_si = feature_cols + ['lgb_oof_IC50', 'lgb_oof_CC50']
lgb_oof_predictions_si = np.zeros(len(train_final))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols_lgb_si], train_final.loc[train_idx, 'SI'],
        eval_set=[(train_final.loc[val_idx, feature_cols_lgb_si], train_final.loc[val_idx, 'SI'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    lgb_oof_predictions_si[val_idx] = model.predict(train_final.loc[val_idx, feature_cols_lgb_si])
    lgb_models_si.append(model)

lgb_test_ic50 = np.zeros(len(test))
lgb_test_cc50 = np.zeros(len(test))
lgb_test_si = np.zeros(len(test))

for model in lgb_models_ic50:
    lgb_test_ic50 += model.predict(test[feature_cols]) / 5

for model in lgb_models_cc50:
    lgb_test_cc50 += model.predict(test[feature_cols]) / 5

test_meta_lgb = test.copy()
test_meta_lgb['lgb_oof_IC50'] = lgb_test_ic50
test_meta_lgb['lgb_oof_CC50'] = lgb_test_cc50

for model in lgb_models_si:
    lgb_test_si += model.predict(test_meta_lgb[feature_cols_lgb_si]) / 5

print("Обучение LightGBM завершено")
print(f"Локальный OOF RMSE для IC50 (LGB): {root_mean_squared_error(train_final['IC50, mM'], train_final['lgb_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (LGB): {root_mean_squared_error(train_final['CC50, mM'], train_final['lgb_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (LGB): {root_mean_squared_error(train_final['SI'], lgb_oof_predictions_si):.4f}")


Обучение LightGBM завершено
Локальный OOF RMSE для IC50 (LGB): 328.7775
Локальный OOF RMSE для CC50 (LGB): 456.3897
Локальный OOF RMSE для SI (LGB): 9.5069


In [15]:
triple_blended_ic50 = (test_preds_ic50 + xgb_test_ic50 + lgb_test_ic50) / 3
triple_blended_cc50 = (test_preds_cc50 + xgb_test_cc50 + lgb_test_cc50) / 3
triple_blended_si = (test_preds_si + xgb_test_si + lgb_test_si) / 3

submission_triple = pd.DataFrame({
    'index': test['index'],
    'IC50': triple_blended_ic50,
    'CC50': triple_blended_cc50,
    'SI': triple_blended_si
})

submission_triple.to_csv('submission_triple_blend.csv', index=False)

print("Файл submission_triple_blend.csv создан")
print(submission_triple.head())


Файл submission_triple_blend.csv создан
   index        IC50        CC50        SI
0      0  191.264482  397.777092  8.415824
1      1  241.069216  398.769699  5.985821
2      2  157.073158  412.768885  8.002162
3      3  306.050083  410.405055  5.950809
4      4  209.663371  326.389420  5.233820


In [ ]:
warnings.filterwarnings('ignore')
def objective_catboost_ic50(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1200),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_seed': 42,
        'verbose': 0
    }

    fold_errors = []

    for fold in range(5):
        train_idx = train_final[train_final['fold'] != fold].index
        val_idx = train_final[train_final['fold'] == fold].index

        X_train = train_final.loc[train_idx, feature_cols]
        y_train = train_final.loc[train_idx, 'IC50, mM']
        X_val = train_final.loc[val_idx, feature_cols]
        y_val = train_final.loc[val_idx, 'IC50, mM']

        model = CatBoostRegressor(**params)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)

        preds = model.predict(X_val)
        fold_rmse = root_mean_squared_error(y_val, preds)
        fold_errors.append(fold_rmse)


    return np.mean(fold_errors)

study = optuna.create_study(direction='minimize')

print(" тюнинг гиперпараметров через Optuna")
study.optimize(objective_catboost_ic50, n_trials=10)

print("\n завершено")
print(f"Лучший локальный RMSE для IC50: {study.best_value:.4f}")
print("Идеальные параметры:", study.best_params)


[I 2026-05-24 21:42:27,805] A new study created in memory with name: no-name-c5cf74fc-7693-456c-ad87-42df2e436053


 тюнинг гиперпараметров через Optuna


[I 2026-05-24 21:42:57,457] Trial 0 finished with value: 328.5936124382264 and parameters: {'iterations': 656, 'depth': 8, 'learning_rate': 0.028959119630971728, 'l2_leaf_reg': 1.3784878632854463}. Best is trial 0 with value: 328.5936124382264.
[I 2026-05-24 21:43:11,452] Trial 1 finished with value: 323.144056813462 and parameters: {'iterations': 796, 'depth': 6, 'learning_rate': 0.03382517395382496, 'l2_leaf_reg': 2.864190270459776}. Best is trial 1 with value: 323.144056813462.
[I 2026-05-24 21:43:29,150] Trial 2 finished with value: 324.0418956375214 and parameters: {'iterations': 764, 'depth': 7, 'learning_rate': 0.04657278553818172, 'l2_leaf_reg': 3.2824456578086494}. Best is trial 1 with value: 323.144056813462.
[I 2026-05-24 21:43:34,438] Trial 3 finished with value: 323.8384263116259 and parameters: {'iterations': 574, 'depth': 5, 'learning_rate': 0.061530705840539254, 'l2_leaf_reg': 4.266625037429767}. Best is trial 1 with value: 323.144056813462.
[I 2026-05-24 21:43:39,923] 


 завершено
Лучший локальный RMSE для IC50: 322.2557
Идеальные параметры: {'iterations': 740, 'depth': 6, 'learning_rate': 0.07962111841819117, 'l2_leaf_reg': 5.520998238498503}


In [21]:

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1200),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0), # L2 регуляризация
        'random_state': 42,
        'n_jobs': -1
    }
    errors = []
    for fold in range(5):
        train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index
        model = XGBRegressor(**params, early_stopping_rounds=50)
        model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
                  eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], verbose=False)
        errors.append(root_mean_squared_error(train_final.loc[val_idx, 'IC50, mM'], model.predict(train_final.loc[val_idx, feature_cols])))
    return np.mean(errors)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=10)
print(f"Лучший RMSE для XGBoost: {study_xgb.best_value:.4f}")

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1200),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    errors = []
    for fold in range(5):
        train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index
        model = LGBMRegressor(**params)
        model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
                  eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
                  callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
        errors.append(root_mean_squared_error(train_final.loc[val_idx, 'IC50, mM'], model.predict(train_final.loc[val_idx, feature_cols])))
    return np.mean(errors)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=10)
print(f"Лучший RMSE для LightGBM: {study_lgb.best_value:.4f}")


[I 2026-05-24 21:50:14,145] A new study created in memory with name: no-name-55e6f480-7671-40fa-939b-76795027b656
[I 2026-05-24 21:50:17,122] Trial 0 finished with value: 323.7189539312156 and parameters: {'n_estimators': 1003, 'max_depth': 4, 'learning_rate': 0.022708732518232577, 'reg_lambda': 7.485047250483183}. Best is trial 0 with value: 323.7189539312156.
[I 2026-05-24 21:50:20,664] Trial 1 finished with value: 324.511252021245 and parameters: {'n_estimators': 699, 'max_depth': 8, 'learning_rate': 0.09626911188973762, 'reg_lambda': 1.287526022687429}. Best is trial 0 with value: 323.7189539312156.
[I 2026-05-24 21:50:22,263] Trial 2 finished with value: 320.63552420731764 and parameters: {'n_estimators': 655, 'max_depth': 4, 'learning_rate': 0.08548015173566016, 'reg_lambda': 7.940864922300598}. Best is trial 2 with value: 320.63552420731764.
[I 2026-05-24 21:50:30,568] Trial 3 finished with value: 325.0589098800903 and parameters: {'n_estimators': 935, 'max_depth': 8, 'learning_

Лучший RMSE для XGBoost: 320.6355


[I 2026-05-24 21:50:51,088] Trial 0 finished with value: 323.21000220473246 and parameters: {'n_estimators': 789, 'max_depth': 7, 'learning_rate': 0.030215349483860347, 'reg_lambda': 4.506055564843539}. Best is trial 0 with value: 323.21000220473246.
[I 2026-05-24 21:50:51,616] Trial 1 finished with value: 322.43116024553194 and parameters: {'n_estimators': 1130, 'max_depth': 6, 'learning_rate': 0.03939840825181381, 'reg_lambda': 6.813267226737738}. Best is trial 1 with value: 322.43116024553194.
[I 2026-05-24 21:50:52,005] Trial 2 finished with value: 323.0529649875572 and parameters: {'n_estimators': 1079, 'max_depth': 6, 'learning_rate': 0.08367946570698667, 'reg_lambda': 9.849980182187464}. Best is trial 1 with value: 322.43116024553194.
[I 2026-05-24 21:50:52,540] Trial 3 finished with value: 321.91562317672197 and parameters: {'n_estimators': 678, 'max_depth': 6, 'learning_rate': 0.09492621445153143, 'reg_lambda': 9.874662835932675}. Best is trial 3 with value: 321.91562317672197

Лучший RMSE для LightGBM: 319.6886


In [22]:
cb_params = {'iterations': 740, 'depth': 6, 'learning_rate': 0.0796, 'l2_leaf_reg': 5.52, 'random_seed': 42, 'verbose': 0}
xgb_params = {'n_estimators': 655, 'max_depth': 4, 'learning_rate': 0.0855, 'reg_lambda': 7.94, 'random_state': 42, 'n_jobs': -1}
lgb_params = {'n_estimators': 875, 'max_depth': 6, 'learning_rate': 0.0449, 'reg_lambda': 2.22, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}

opt_test_ic50_cb, opt_test_ic50_xgb, opt_test_ic50_lgb = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))
opt_test_cc50_cb, opt_test_cc50_xgb, opt_test_cc50_lgb = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))
opt_test_si_cb, opt_test_si_xgb, opt_test_si_lgb = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))

train_final['opt_oof_IC50'] = 0.0
train_final['opt_xgb_oof_IC50'] = 0.0
train_final['opt_lgb_oof_IC50'] = 0.0

train_final['opt_oof_CC50'] = 0.0
train_final['opt_xgb_oof_CC50'] = 0.0
train_final['opt_lgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    # CatBoost
    m_cb = CatBoostRegressor(**cb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']), early_stopping_rounds=50, verbose=False)
    train_final.loc[val_idx, 'opt_oof_IC50'] = m_cb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_ic50_cb += m_cb.predict(test[feature_cols]) / 5

    # XGBoost
    m_xgb = XGBRegressor(**xgb_params, early_stopping_rounds=50).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'opt_xgb_oof_IC50'] = m_xgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_ic50_xgb += m_xgb.predict(test[feature_cols]) / 5

    # LightGBM
    m_lgb = LGBMRegressor(**lgb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    train_final.loc[val_idx, 'opt_lgb_oof_IC50'] = m_lgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_ic50_lgb += m_lgb.predict(test[feature_cols]) / 5

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    m_cb = CatBoostRegressor(**cb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']), early_stopping_rounds=50, verbose=False)
    train_final.loc[val_idx, 'opt_oof_CC50'] = m_cb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_cc50_cb += m_cb.predict(test[feature_cols]) / 5

    m_xgb = XGBRegressor(**xgb_params, early_stopping_rounds=50).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'opt_xgb_oof_CC50'] = m_xgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_cc50_xgb += m_xgb.predict(test[feature_cols]) / 5

    m_lgb = LGBMRegressor(**lgb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    train_final.loc[val_idx, 'opt_lgb_oof_CC50'] = m_lgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_cc50_lgb += m_lgb.predict(test[feature_cols]) / 5

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    f_cb = feature_cols + ['opt_oof_IC50', 'opt_oof_CC50']
    f_xgb = feature_cols + ['opt_xgb_oof_IC50', 'opt_xgb_oof_CC50']
    f_lgb = feature_cols + ['opt_lgb_oof_IC50', 'opt_lgb_oof_CC50']

    t_cb, t_xgb, t_lgb = test.copy(), test.copy(), test.copy()
    t_cb['opt_oof_IC50'], t_cb['opt_oof_CC50'] = opt_test_ic50_cb, opt_test_cc50_cb
    t_xgb['opt_xgb_oof_IC50'], t_xgb['opt_xgb_oof_CC50'] = opt_test_ic50_xgb, opt_test_cc50_xgb
    t_lgb['opt_lgb_oof_IC50'], t_lgb['opt_lgb_oof_CC50'] = opt_test_ic50_lgb, opt_test_cc50_lgb


    m_cb = CatBoostRegressor(**cb_params).fit(train_final.loc[train_idx, f_cb], train_final.loc[train_idx, 'SI'], eval_set=(train_final.loc[val_idx, f_cb], train_final.loc[val_idx, 'SI']), early_stopping_rounds=50, verbose=False)
    opt_test_si_cb += m_cb.predict(t_cb[f_cb]) / 5

    m_xgb = XGBRegressor(**xgb_params, early_stopping_rounds=50).fit(train_final.loc[train_idx, f_xgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_xgb], train_final.loc[val_idx, 'SI'])], verbose=False)
    opt_test_si_xgb += m_xgb.predict(t_xgb[f_xgb]) / 5

    m_lgb = LGBMRegressor(**lgb_params).fit(train_final.loc[train_idx, f_lgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_lgb], train_final.loc[val_idx, 'SI'])], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    opt_test_si_lgb += m_lgb.predict(t_lgb[f_lgb]) / 5

final_ic50 = (opt_test_ic50_cb + opt_test_ic50_xgb + opt_test_ic50_lgb) / 3
final_cc50 = (opt_test_cc50_cb + opt_test_cc50_xgb + opt_test_cc50_lgb) / 3
final_si   = (opt_test_si_cb + opt_test_si_xgb + opt_test_si_lgb) / 3

submission_optuna = pd.DataFrame({
    'index': test['index'],
    'IC50': final_ic50,
    'CC50': final_cc50,
    'SI': final_si
})

submission_optuna.to_csv('submission_optuna_blend.csv', index=False)
print("файл submission_optuna_blend.csv создан")
print(submission_optuna.head())


файл submission_optuna_blend.csv создан
   index        IC50        CC50        SI
0      0  192.747669  402.152946  7.944375
1      1  245.899287  408.238617  5.487991
2      2  139.849060  428.309071  7.719802
3      3  270.212158  408.904417  5.910707
4      4  218.027337  329.315638  4.794091


In [ ]:

train_final['ridge_oof_IC50'] = 0.0
train_final['ridge_oof_CC50'] = 0.0
ridge_oof_predictions_si = np.zeros(len(train_final))

ridge_models_ic50, ridge_scalers_ic50 = [], []
ridge_models_cc50, ridge_scalers_cc50 = [], []
ridge_models_si, ridge_scalers_si = [], []

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_final.loc[train_idx, feature_cols])
    X_val_scaled = scaler.transform(train_final.loc[val_idx, feature_cols])

    model = Ridge(alpha=10.0, random_state=42)
    model.fit(X_train_scaled, train_final.loc[train_idx, 'IC50, mM'])

    train_final.loc[val_idx, 'ridge_oof_IC50'] = model.predict(X_val_scaled)
    ridge_models_ic50.append(model)
    ridge_scalers_ic50.append(scaler)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_final.loc[train_idx, feature_cols])
    X_val_scaled = scaler.transform(train_final.loc[val_idx, feature_cols])

    model = Ridge(alpha=10.0, random_state=42)
    model.fit(X_train_scaled, train_final.loc[train_idx, 'CC50, mM'])

    train_final.loc[val_idx, 'ridge_oof_CC50'] = model.predict(X_val_scaled)
    ridge_models_cc50.append(model)
    ridge_scalers_cc50.append(scaler)

feature_cols_ridge_si = feature_cols + ['ridge_oof_IC50', 'ridge_oof_CC50']

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_final.loc[train_idx, feature_cols_ridge_si])
    X_val_scaled = scaler.transform(train_final.loc[val_idx, feature_cols_ridge_si])

    model = Ridge(alpha=10.0, random_state=42)
    model.fit(X_train_scaled, train_final.loc[train_idx, 'SI'])

    ridge_oof_predictions_si[val_idx] = model.predict(X_val_scaled)
    ridge_models_si.append(model)
    ridge_scalers_si.append(scaler)

ridge_test_ic50 = np.zeros(len(test))
ridge_test_cc50 = np.zeros(len(test))
ridge_test_si = np.zeros(len(test))

for model, scaler in zip(ridge_models_ic50, ridge_scalers_ic50):
    ridge_test_ic50 += model.predict(scaler.transform(test[feature_cols])) / 5

for model, scaler in zip(ridge_models_cc50, ridge_scalers_cc50):
    ridge_test_cc50 += model.predict(scaler.transform(test[feature_cols])) / 5

test_meta_ridge = test.copy()
test_meta_ridge['ridge_oof_IC50'] = ridge_test_ic50
test_meta_ridge['ridge_oof_CC50'] = ridge_test_cc50

for model, scaler in zip(ridge_models_si, ridge_scalers_si):
    ridge_test_si += model.predict(scaler.transform(test_meta_ridge[feature_cols_ridge_si])) / 5

print("Обучение Ridge-регрессии завершено")
print(f"Локальный OOF RMSE для IC50 (Ridge): {root_mean_squared_error(train_final['IC50, mM'], train_final['ridge_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (Ridge): {root_mean_squared_error(train_final['CC50, mM'], train_final['ridge_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (Ridge): {root_mean_squared_error(train_final['SI'], ridge_oof_predictions_si):.4f}")


In [42]:
def generate_chemical_features(df):
    df_copy = df.copy()
    eps = 1e-5
    df_copy['custom_MolWt_per_Atom'] = df_copy['MolWt'] / (df_copy['HeavyAtomCount'] + eps)
    df_copy['custom_Polar_Ratio'] = df_copy['TPSA'] / (df_copy['LabuteASA'] + eps)
    df_copy['custom_Hydrophobic_Efficiency'] = df_copy['MolLogP'] / (df_copy['LabuteASA'] + eps)
    df_copy['custom_Electrons_per_Atom'] = df_copy['NumValenceElectrons'] / (df_copy['HeavyAtomCount'] + eps)
    return df_copy

train_final = generate_chemical_features(train_final)
test = generate_chemical_features(test)

new_features = ['custom_MolWt_per_Atom', 'custom_Polar_Ratio', 'custom_Hydrophobic_Efficiency', 'custom_Electrons_per_Atom']
feature_cols = feature_cols + new_features

print(f"Новое количество признаков в feature_cols: {len(feature_cols)}")

Новое количество признаков в feature_cols: 169


In [ ]:
feature_cols = list(set(feature_cols))
oof_cols_to_clear = ['oof_IC50', 'oof_CC50', 'xgb_oof_IC50', 'xgb_oof_CC50', 'lgb_oof_IC50', 'lgb_oof_CC50', 'ridge_oof_IC50', 'ridge_oof_CC50']
for col in oof_cols_to_clear:
    train_final[col] = 0.0

test_preds_ic50, xgb_test_ic50, lgb_test_ic50, ridge_test_ic50 = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))
test_preds_cc50, xgb_test_cc50, lgb_test_cc50, ridge_test_cc50 = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    m_cb_ic = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']), early_stopping_rounds=100, verbose=False)
    train_final.loc[val_idx, 'oof_IC50'] = m_cb_ic.predict(train_final.loc[val_idx, feature_cols])
    test_preds_ic50 += m_cb_ic.predict(test[feature_cols]) / 5

    m_cb_cc = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']), early_stopping_rounds=100, verbose=False)
    train_final.loc[val_idx, 'oof_CC50'] = m_cb_cc.predict(train_final.loc[val_idx, feature_cols])
    test_preds_cc50 += m_cb_cc.predict(test[feature_cols]) / 5

    m_xgb_ic = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'xgb_oof_IC50'] = m_xgb_ic.predict(train_final.loc[val_idx, feature_cols])
    xgb_test_ic50 += m_xgb_ic.predict(test[feature_cols]) / 5

    m_xgb_cc = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'xgb_oof_CC50'] = m_xgb_cc.predict(train_final.loc[val_idx, feature_cols])
    xgb_test_cc50 += m_xgb_cc.predict(test[feature_cols]) / 5

    m_lgb_ic = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    train_final.loc[val_idx, 'lgb_oof_IC50'] = m_lgb_ic.predict(train_final.loc[val_idx, feature_cols])
    lgb_test_ic50 += m_lgb_ic.predict(test[feature_cols]) / 5

    m_lgb_cc = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    train_final.loc[val_idx, 'lgb_oof_CC50'] = m_lgb_cc.predict(train_final.loc[val_idx, feature_cols])
    lgb_test_cc50 += m_lgb_cc.predict(test[feature_cols]) / 5

    imp, scl = SimpleImputer(strategy='median'), StandardScaler()
    X_tr_scaled = scl.fit_transform(imp.fit_transform(train_final.loc[train_idx, feature_cols]))
    X_va_scaled = scl.transform(imp.transform(train_final.loc[val_idx, feature_cols]))
    t_imp_scaled = scl.transform(imp.transform(test[feature_cols]))

    m_rg_ic = Ridge(alpha=10.0, random_state=42).fit(X_tr_scaled, train_final.loc[train_idx, 'IC50, mM'])
    train_final.loc[val_idx, 'ridge_oof_IC50'] = m_rg_ic.predict(X_va_scaled)
    ridge_test_ic50 += m_rg_ic.predict(t_imp_scaled) / 5

    m_rg_cc = Ridge(alpha=10.0, random_state=42).fit(X_tr_scaled, train_final.loc[train_idx, 'CC50, mM'])
    train_final.loc[val_idx, 'ridge_oof_CC50'] = m_rg_cc.predict(X_va_scaled)
    ridge_test_cc50 += m_rg_cc.predict(t_imp_scaled) / 5

test_preds_si, xgb_test_si, lgb_test_si, ridge_test_si = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    f_cb, f_xgb, f_lgb, f_rg = feature_cols + ['oof_IC50', 'oof_CC50'], feature_cols + ['xgb_oof_IC50', 'xgb_oof_CC50'], feature_cols + ['lgb_oof_IC50', 'lgb_oof_CC50'], feature_cols + ['ridge_oof_IC50', 'ridge_oof_CC50']
    t_cb, t_xgb, t_lgb, t_rg = test.copy(), test.copy(), test.copy(), test.copy()
    t_cb['oof_IC50'], t_cb['oof_CC50'] = test_preds_ic50, test_preds_cc50
    t_xgb['xgb_oof_IC50'], t_xgb['xgb_oof_CC50'] = xgb_test_ic50, xgb_test_cc50
    t_lgb['lgb_oof_IC50'], t_lgb['lgb_oof_CC50'] = lgb_test_ic50, lgb_test_cc50
    t_rg['ridge_oof_IC50'], t_rg['ridge_oof_CC50'] = ridge_test_ic50, ridge_test_cc50

    m_cb = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_final.loc[train_idx, f_cb], train_final.loc[train_idx, 'SI'], eval_set=(train_final.loc[val_idx, f_cb], train_final.loc[val_idx, 'SI']), early_stopping_rounds=100, verbose=False)
    test_preds_si += m_cb.predict(t_cb[f_cb]) / 5

    m_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_final.loc[train_idx, f_xgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_xgb], train_final.loc[val_idx, 'SI'])], verbose=False)
    xgb_test_si += m_xgb.predict(t_xgb[f_xgb]) / 5

    m_lgb = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_final.loc[train_idx, f_lgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_lgb], train_final.loc[val_idx, 'SI'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    lgb_test_si += m_lgb.predict(t_lgb[f_lgb]) / 5

    X_tr_sc = scl.fit_transform(imp.fit_transform(train_final.loc[train_idx, f_rg]))
    ridge_test_si += Ridge(alpha=10.0, random_state=42).fit(X_tr_sc, train_final.loc[train_idx, 'SI']).predict(scl.transform(imp.transform(t_rg[f_rg]))) / 5

final_blend_ic50 = 0.283 * test_preds_ic50 + 0.283 * xgb_test_ic50 + 0.283 * lgb_test_ic50 + 0.151 * ridge_test_ic50
final_blend_cc50 = 0.283 * test_preds_cc50 + 0.283 * xgb_test_cc50 + 0.283 * lgb_test_cc50 + 0.151 * ridge_test_cc50
final_blend_si   = 0.283 * test_preds_si   + 0.283 * xgb_test_si   + 0.283 * lgb_test_si   + 0.151 * ridge_test_si

submission_fe_blend = pd.DataFrame({
    'index': test['index'],
    'IC50': final_blend_ic50,
    'CC50': final_blend_cc50,
    'SI': final_blend_si
})

submission_fe_blend.to_csv('submission_fe_blend.csv', index=False)
print(submission_fe_blend.head())

   index        IC50        CC50        SI
0      0  182.363373  312.399854  7.519580
1      1  263.684480  418.167567  6.441381
2      2  146.206765  464.473178  7.921809
3      3  267.152327  383.956610  5.913924
4      4  217.402905  350.176027  4.600513


In [ ]:
feature_cols_custom_si = feature_cols + ['oof_IC50', 'oof_CC50']

global current_fold_cc50, current_fold_ic50

def physics_anchored_loss(y_true, y_pred, cc50_oof, ic50_oof, gamma=0.2):
    si_theoretical = cc50_oof / (ic50_oof + 1e-5)
    error_true = y_pred - y_true
    error_physics = y_pred - si_theoretical
    gradient = 2 * error_true + 2 * gamma * error_physics
    hessian = np.full_like(y_true, 2 + 2 * gamma)
    return gradient, hessian

def lgb_custom_objective(y_pred, dataset):
    y_true = dataset.get_label()
    global current_fold_cc50, current_fold_ic50
    grad, hess = physics_anchored_loss(y_true, y_pred, current_fold_cc50, current_fold_ic50, gamma=0.2)
    return grad, hess

custom_lgb_oof_si = np.zeros(len(train_final))
custom_lgb_test_si = np.zeros(len(test))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    X_train = train_final.loc[train_idx, feature_cols_custom_si]
    y_train = train_final.loc[train_idx, 'SI']
    X_val = train_final.loc[val_idx, feature_cols_custom_si]
    y_val = train_final.loc[val_idx, 'SI']

    current_fold_cc50 = train_final.loc[train_idx, 'oof_CC50'].values
    current_fold_ic50 = train_final.loc[train_idx, 'oof_IC50'].values

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

    params = {
        'objective': lgb_custom_objective,
        'metric': 'rmse',
        'learning_rate': 0.05,
        'max_depth': 6,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=1000,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    custom_lgb_oof_si[val_idx] = model.predict(X_val)

    t_meta = test.copy()
    t_meta['oof_IC50'] = test_preds_ic50
    t_meta['oof_CC50'] = test_preds_cc50
    custom_lgb_test_si += model.predict(t_meta[feature_cols_custom_si]) / 5

In [46]:
final_physics_si = 0.283 * test_preds_si + 0.283 * xgb_test_si + 0.283 * custom_lgb_test_si + 0.151 * ridge_test_si

submission_physics_blend = pd.DataFrame({
    'index': test['index'],
    'IC50': final_blend_ic50,
    'CC50': final_blend_cc50,
    'SI': final_physics_si
})

submission_physics_blend.to_csv('submission_physics_blend.csv', index=False)
print(submission_physics_blend.head())

   index        IC50        CC50        SI
0      0  182.363373  312.399854  7.051302
1      1  263.684480  418.167567  5.864131
2      2  146.206765  464.473178  7.325474
3      3  267.152327  383.956610  5.462690
4      4  217.402905  350.176027  3.856418
